# Executive Summary

- this project analyzes ecommerce sales, marketing channel performance, discount effectiveness, and inventory allocation across six countries. The objective was to identify opportunities to improve marketing efficiency and profitability. Analysis was performed using Python for data preparation and statistical testing and Tableau for visualization.

Key findings include:

- website Banner and App Mobile generated over 96% of total revenue.
- higher discounts significantly reduced profit margins without increasing purchase quantity.
- germany generated the highest sales volume despite substantially lower recorded inventory availability than France.

# Business Problem

the business wants to understand whether current marketing activities, discount strategies, and inventory allocation support profitable sales growth.

the project focuses on identifying:

- which marketing channels generate the strongest sales performance.
- whether discounts increase purchasing behaviour enough to justify lower margins.
- Whether inventory allocation aligns with observed demand patterns.

# Dataset Overview

Period:
April 2025 – June 2025 (~75 days)

Scope:
- 2253 sales items
- 905 sales transactions
- 500 products
- 1000 customers
- 6 countries

Limitations:
- No marketing spend data available
- No inventory history
- No logistics information 
- Short observation period (~75 days)

# Data Cleaning & Preparation

- merged sales, customers, products, inventory, and campaign tables.
- created revenue metric.
- created profit metric.
- created profit margin percentage.
- validated sales totals.
- created discount group segmentation.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
campaign = pd.read_csv('dataset_fashion_store_campaigns.csv')
customer = pd.read_csv('dataset_fashion_store_customers.csv')
product = pd.read_csv('dataset_fashion_store_products.csv')
sales = pd.read_csv('dataset_fashion_store_sales.csv')
sales_item = pd.read_csv('dataset_fashion_store_salesitems.csv')
stock = pd.read_csv('dataset_fashion_store_stock.csv') 

In [3]:
for name, df in {'campaign': campaign,
                 'customer': customer,
                 'product': product,
                 'sales': sales,
                 'sales_item': sales_item,
                 'stock': stock}.items():
    print('\n')
    print(name.upper())
    print(df.shape)
    print(df.columns.tolist())



CAMPAIGN
(7, 7)
['campaign_id', 'campaign_name', 'start_date', 'end_date', 'channel', 'discount_type', 'discount_value']


CUSTOMER
(1000, 4)
['customer_id', 'country', 'age_range', 'signup_date']


PRODUCT
(500, 9)
['product_id', 'product_name', 'category', 'brand', 'color', 'size', 'catalog_price', 'cost_price', 'gender']


SALES
(905, 7)
['sale_id', 'channel', 'discounted', 'total_amount', 'sale_date', 'customer_id', 'country']


SALES_ITEM
(2253, 13)
['item_id', 'sale_id', 'product_id', 'quantity', 'original_price', 'unit_price', 'discount_applied', 'discount_percent', 'discounted', 'item_total', 'sale_date', 'channel', 'channel_campaigns']


STOCK
(1000, 3)
['country', 'product_id', 'stock_quantity']


In [4]:
campaign

,campaign_id,campaign_name,start_date,end_date,channel,discount_type,discount_value
0,1,Spring Flash Sale,2025-04-01,2025-04-07,Email,Percentage,10.00%
1,2,Easter Promotion,2025-04-08,2025-04-15,Social Media,Fixed,10
2,3,Mother's Day Campaign,2025-05-01,2025-05-09,Email,Percentage,25.00%
3,4,Mid-Season Clearance,2025-05-10,2025-05-19,App Mobile,Percentage,30.00%
4,5,TIVA Week,2025-05-20,2025-05-31,Social Media,Percentage,30.00%
5,6,June Price Drop,2025-06-01,2025-06-09,Website Banner,Percentage,10.00%
6,7,Early Summer Deals,2025-06-10,2025-06-17,Email,Fixed,12


campaign table feeds me with

1- campagin id ---> for merging 

2- compagin date ---> sales generating, customer acquisition (signing up) could be potenially how many of signed up client made a move. 

3- the channel of the compaign 

4- either fixed discount or a percentage. 

In [5]:
customer

,customer_id,country,age_range,signup_date
0,1,France,56-65,2025-04-24
1,2,France,36-45,2025-02-24
2,3,Netherlands,46-55,2024-04-12
3,4,Italy,36-45,2025-03-11
4,5,Spain,26-35,2025-04-26
...,...,...,...,...
995,996,Germany,16-25,2025-05-08
996,997,Italy,56-65,2025-03-29
997,998,France,36-45,2025-01-07
998,999,Germany,56-65,2025-02-21


customer feeds:

1- id for each customer ---> sorting 

2- country ---> 6 countries.

3- age range ---> 5 age groupe

4- sign up date ----> could tell the effectiviness of compaign on clients signing up.

In [6]:
customer['country'].unique()

array(['France', 'Netherlands', 'Italy', 'Spain', 'Germany', 'Portugal'],
      dtype=object)

In [7]:
customer['age_range'].unique()

array(['56-65', '36-45', '46-55', '26-35', '16-25'], dtype=object)

In [8]:
product

,product_id,product_name,category,brand,color,size,catalog_price,cost_price,gender
0,1,Soft Wrap Dress,Dresses,Tiva,Green,S,40.41,20.70,Female
1,2,Soft Wrap Tee,T-Shirts,Tiva,White,S,78.45,53.76,Female
2,3,Soft Linen Tee,T-Shirts,Tiva,Green,XL,23.90,14.81,Female
3,4,Soft Ribbed Tee,T-Shirts,Tiva,White,S,60.00,34.78,Female
4,5,Soft Wrap Trousers,Pants,Tiva,Blue,M,36.84,16.46,Female
...,...,...,...,...,...,...,...,...,...
495,496,Tailored High-Waist Trousers,Pants,Tiva,Black,S,54.61,27.63,Female
496,497,Dresses Drop 8,Dresses,Tiva,Black,L,36.05,20.73,Female
497,498,T-Shirts Drop 8,T-Shirts,Tiva,White,L,38.33,16.23,Female
498,499,Sleepwear Drop 4,Sleepwear,Tiva,Green,M,30.07,20.82,Female


product: 

1- id and name ---> useful for grouping

2- category ---> categrisation 

3- brand ---> could be a specific dominnace of one brand that generate most of the sales.

4- cataloug price and cost price ----> the difference generate profit if marketing and operational cost included in catalog price.

5- gender ---> female store.

In [9]:
product['gender'].nunique()

1

In [10]:
sales

,sale_id,channel,discounted,total_amount,sale_date,customer_id,country
0,10,E-commerce,0,299.70,2025-05-21,195,France
1,100,App Mobile,0,681.05,2025-04-21,518,Germany
2,1000,E-commerce,0,324.50,2025-05-20,439,Germany
3,1001,E-commerce,0,287.85,2025-04-05,349,Germany
4,1003,App Mobile,0,430.64,2025-06-06,727,Portugal
...,...,...,...,...,...,...,...
900,992,App Mobile,1,214.08,2025-05-13,375,Germany
901,993,E-commerce,0,311.37,2025-04-21,99,Spain
902,994,App Mobile,1,477.09,2025-05-15,798,Portugal
903,995,E-commerce,0,489.02,2025-05-19,565,Spain


sales:

1- sale id

2- channel ---> could be linked to channel in compaign

3- discount ---> discount bolean values.

4- total amount ---> potenially equal to catalog price. 

5- sale date ---> organically or because of compaign

6- customer id ---> for linkeage

In [11]:
sales['country'].unique()

array(['France', 'Germany', 'Portugal', 'Spain', 'Italy', 'Netherlands'],
      dtype=object)

In [12]:
sales_item

,item_id,sale_id,product_id,quantity,original_price,unit_price,discount_applied,discount_percent,discounted,item_total,sale_date,channel,channel_campaigns
0,2270,658,403,1,81.80,81.80,0.00,0.00%,0,81.80,2025-06-16,App Mobile,App Mobile
1,1170,336,284,1,81.79,81.79,0.00,0.00%,0,81.79,2025-06-17,E-commerce,Website Banner
2,2496,1255,71,1,80.76,80.76,0.00,0.00%,0,80.76,2025-04-16,App Mobile,App Mobile
3,1273,331,98,1,78.52,78.52,0.00,0.00%,0,78.52,2025-05-06,App Mobile,App Mobile
4,1829,1079,98,1,78.52,78.52,0.00,0.00%,0,78.52,2025-06-15,App Mobile,App Mobile
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2248,516,1300,334,5,20.82,20.82,0.00,0.00%,0,104.10,2025-04-15,E-commerce,Website Banner
2249,1240,727,334,5,20.82,20.82,0.00,0.00%,0,104.10,2025-04-22,E-commerce,Website Banner
2250,2394,172,246,5,28.98,20.29,8.69,30.00%,1,101.43,2025-05-13,App Mobile,App Mobile
2251,1631,986,195,5,16.69,16.69,0.00,0.00%,0,83.45,2025-05-23,App Mobile,App Mobile


In [13]:
sales_item['sale_id'].nunique()

905

In [14]:
sales['sale_id'].nunique()

905

In [15]:
stock

,country,product_id,stock_quantity
0,France,1,61
1,France,2,24
2,France,3,81
3,France,4,70
4,France,5,30
...,...,...,...
995,Germany,496,1
996,Germany,497,1
997,Germany,498,1
998,Germany,499,1


In [16]:
product['brand'].nunique()

1

# Main Findings - Business Drivers

# H1

Which campaign channel generates the highest revenue and profit?

In [17]:
sales_item_totals = (sales_item.groupby('sale_id')['item_total'].sum().reset_index())
sales_item_totals

,sale_id,item_total
0,2,357.54
1,3,337.56
2,6,416.43
3,8,408.13
4,10,299.70
...,...,...
900,1343,352.50
901,1344,179.43
902,1348,451.03
903,1349,173.50


In [18]:
check = sales.merge(sales_item_totals,on='sale_id',how='left')
check

,sale_id,channel,discounted,total_amount,sale_date,customer_id,country,item_total
0,10,E-commerce,0,299.70,2025-05-21,195,France,299.70
1,100,App Mobile,0,681.05,2025-04-21,518,Germany,681.05
2,1000,E-commerce,0,324.50,2025-05-20,439,Germany,324.50
3,1001,E-commerce,0,287.85,2025-04-05,349,Germany,287.85
4,1003,App Mobile,0,430.64,2025-06-06,727,Portugal,430.64
...,...,...,...,...,...,...,...,...
900,992,App Mobile,1,214.08,2025-05-13,375,Germany,214.08
901,993,E-commerce,0,311.37,2025-04-21,99,Spain,311.37
902,994,App Mobile,1,477.09,2025-05-15,798,Portugal,477.09
903,995,E-commerce,0,489.02,2025-05-19,565,Spain,489.02


In [19]:
check['difference'] = (check['total_amount'] - check['item_total'])

check['difference'].describe()

count    9.050000e+02
mean    -9.107509e-16
std      2.622348e-14
min     -1.136868e-13
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      1.136868e-13
Name: difference, dtype: float64

In [20]:
check['difference'].value_counts().head(20)

difference
 0.000000e+00    724
-5.684342e-14     50
 5.684342e-14     48
-2.842171e-14     30
 2.842171e-14     27
-1.136868e-13     13
 1.136868e-13      7
 1.421085e-14      5
-1.421085e-14      1
Name: count, dtype: int64

In [21]:
check['difference'].eq(0).mean()

np.float64(0.8)

In [22]:
master = (sales_item.merge(product, on='product_id', how='left').merge(sales[['sale_id', 'customer_id', 'country']], on='sale_id', how='left').merge(customer[['customer_id', 'age_range', 'signup_date']], on='customer_id', how='left'))
master

,item_id,sale_id,product_id,quantity,original_price,unit_price,discount_applied,discount_percent,discounted,item_total,...,brand,color,size,catalog_price,cost_price,gender,customer_id,country,age_range,signup_date
0,2270,658,403,1,81.80,81.80,0.00,0.00%,0,81.80,...,Tiva,Red,L,81.80,45.12,Female,835,Portugal,46-55,2025-04-26
1,1170,336,284,1,81.79,81.79,0.00,0.00%,0,81.79,...,Tiva,White,35,81.79,35.02,Female,790,France,16-25,2025-04-26
2,2496,1255,71,1,80.76,80.76,0.00,0.00%,0,80.76,...,Tiva,Red,XL,80.76,51.01,Female,464,Germany,36-45,2025-04-14
3,1273,331,98,1,78.52,78.52,0.00,0.00%,0,78.52,...,Tiva,Black,38,78.52,41.48,Female,100,Italy,26-35,2025-01-30
4,1829,1079,98,1,78.52,78.52,0.00,0.00%,0,78.52,...,Tiva,Black,38,78.52,41.48,Female,837,Germany,46-55,2025-03-02
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2248,516,1300,334,5,20.82,20.82,0.00,0.00%,0,104.10,...,Tiva,Black,M,20.82,9.05,Female,147,Germany,36-45,2025-02-13
2249,1240,727,334,5,20.82,20.82,0.00,0.00%,0,104.10,...,Tiva,Black,M,20.82,9.05,Female,834,Netherlands,26-35,2025-04-19
2250,2394,172,246,5,28.98,20.29,8.69,30.00%,1,101.43,...,Tiva,White,XS,28.98,19.66,Female,398,Spain,16-25,2025-01-26
2251,1631,986,195,5,16.69,16.69,0.00,0.00%,0,83.45,...,Tiva,Black,XL,16.69,8.37,Female,735,France,26-35,2025-04-02


In [23]:
master.drop(columns=['brand', 'color', 'size', 'gender', 'signup_date'], inplace=True)
master

,item_id,sale_id,product_id,quantity,original_price,unit_price,discount_applied,discount_percent,discounted,item_total,sale_date,channel,channel_campaigns,product_name,category,catalog_price,cost_price,customer_id,country,age_range
0,2270,658,403,1,81.80,81.80,0.00,0.00%,0,81.80,2025-06-16,App Mobile,App Mobile,Elegant Satin Dress,Dresses,81.80,45.12,835,Portugal,46-55
1,1170,336,284,1,81.79,81.79,0.00,0.00%,0,81.79,2025-06-17,E-commerce,Website Banner,Essential Cotton Shoes,Shoes,81.79,35.02,790,France,16-25
2,2496,1255,71,1,80.76,80.76,0.00,0.00%,0,80.76,2025-04-16,App Mobile,App Mobile,Modern Ribbed Trousers,Pants,80.76,51.01,464,Germany,36-45
3,1273,331,98,1,78.52,78.52,0.00,0.00%,0,78.52,2025-05-06,App Mobile,App Mobile,Modern Boxy Shoes,Shoes,78.52,41.48,100,Italy,26-35
4,1829,1079,98,1,78.52,78.52,0.00,0.00%,0,78.52,2025-06-15,App Mobile,App Mobile,Modern Boxy Shoes,Shoes,78.52,41.48,837,Germany,46-55
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2248,516,1300,334,5,20.82,20.82,0.00,0.00%,0,104.10,2025-04-15,E-commerce,Website Banner,Tailored Ribbed Dress,Dresses,20.82,9.05,147,Germany,36-45
2249,1240,727,334,5,20.82,20.82,0.00,0.00%,0,104.10,2025-04-22,E-commerce,Website Banner,Tailored Ribbed Dress,Dresses,20.82,9.05,834,Netherlands,26-35
2250,2394,172,246,5,28.98,20.29,8.69,30.00%,1,101.43,2025-05-13,App Mobile,App Mobile,Relaxed Satin Dress,Dresses,28.98,19.66,398,Spain,16-25
2251,1631,986,195,5,16.69,16.69,0.00,0.00%,0,83.45,2025-05-23,App Mobile,App Mobile,Vintage Crew Tee,T-Shirts,16.69,8.37,735,France,26-35


In [24]:
master['channel'].value_counts()

channel
E-commerce    1170
App Mobile    1083
Name: count, dtype: int64

In [25]:
pd.crosstab(master['channel_campaigns'], master['channel'])

channel,App Mobile,E-commerce
channel_campaigns,,
App Mobile,963,0
Email,0,19
Social Media,120,0
Website Banner,0,1151


# finding:

- while website banner and app mobile generated most of the sales yet the available dataset is limiting the evaluation of ROI per channel.
  
# limitations

- lack of data regarding campagin cost, impressions, cost per acquisition and click data campaign effectiveness prevent from measuring reliably the effectivness of each channel.


# H2

Discounts increase revenue but may reduce profitability.

In [26]:
master

,item_id,sale_id,product_id,quantity,original_price,unit_price,discount_applied,discount_percent,discounted,item_total,sale_date,channel,channel_campaigns,product_name,category,catalog_price,cost_price,customer_id,country,age_range
0,2270,658,403,1,81.80,81.80,0.00,0.00%,0,81.80,2025-06-16,App Mobile,App Mobile,Elegant Satin Dress,Dresses,81.80,45.12,835,Portugal,46-55
1,1170,336,284,1,81.79,81.79,0.00,0.00%,0,81.79,2025-06-17,E-commerce,Website Banner,Essential Cotton Shoes,Shoes,81.79,35.02,790,France,16-25
2,2496,1255,71,1,80.76,80.76,0.00,0.00%,0,80.76,2025-04-16,App Mobile,App Mobile,Modern Ribbed Trousers,Pants,80.76,51.01,464,Germany,36-45
3,1273,331,98,1,78.52,78.52,0.00,0.00%,0,78.52,2025-05-06,App Mobile,App Mobile,Modern Boxy Shoes,Shoes,78.52,41.48,100,Italy,26-35
4,1829,1079,98,1,78.52,78.52,0.00,0.00%,0,78.52,2025-06-15,App Mobile,App Mobile,Modern Boxy Shoes,Shoes,78.52,41.48,837,Germany,46-55
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2248,516,1300,334,5,20.82,20.82,0.00,0.00%,0,104.10,2025-04-15,E-commerce,Website Banner,Tailored Ribbed Dress,Dresses,20.82,9.05,147,Germany,36-45
2249,1240,727,334,5,20.82,20.82,0.00,0.00%,0,104.10,2025-04-22,E-commerce,Website Banner,Tailored Ribbed Dress,Dresses,20.82,9.05,834,Netherlands,26-35
2250,2394,172,246,5,28.98,20.29,8.69,30.00%,1,101.43,2025-05-13,App Mobile,App Mobile,Relaxed Satin Dress,Dresses,28.98,19.66,398,Spain,16-25
2251,1631,986,195,5,16.69,16.69,0.00,0.00%,0,83.45,2025-05-23,App Mobile,App Mobile,Vintage Crew Tee,T-Shirts,16.69,8.37,735,France,26-35


In [27]:
master = (master.rename(columns={'item_total': 'revenue'}))
master

,item_id,sale_id,product_id,quantity,original_price,unit_price,discount_applied,discount_percent,discounted,revenue,sale_date,channel,channel_campaigns,product_name,category,catalog_price,cost_price,customer_id,country,age_range
0,2270,658,403,1,81.80,81.80,0.00,0.00%,0,81.80,2025-06-16,App Mobile,App Mobile,Elegant Satin Dress,Dresses,81.80,45.12,835,Portugal,46-55
1,1170,336,284,1,81.79,81.79,0.00,0.00%,0,81.79,2025-06-17,E-commerce,Website Banner,Essential Cotton Shoes,Shoes,81.79,35.02,790,France,16-25
2,2496,1255,71,1,80.76,80.76,0.00,0.00%,0,80.76,2025-04-16,App Mobile,App Mobile,Modern Ribbed Trousers,Pants,80.76,51.01,464,Germany,36-45
3,1273,331,98,1,78.52,78.52,0.00,0.00%,0,78.52,2025-05-06,App Mobile,App Mobile,Modern Boxy Shoes,Shoes,78.52,41.48,100,Italy,26-35
4,1829,1079,98,1,78.52,78.52,0.00,0.00%,0,78.52,2025-06-15,App Mobile,App Mobile,Modern Boxy Shoes,Shoes,78.52,41.48,837,Germany,46-55
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2248,516,1300,334,5,20.82,20.82,0.00,0.00%,0,104.10,2025-04-15,E-commerce,Website Banner,Tailored Ribbed Dress,Dresses,20.82,9.05,147,Germany,36-45
2249,1240,727,334,5,20.82,20.82,0.00,0.00%,0,104.10,2025-04-22,E-commerce,Website Banner,Tailored Ribbed Dress,Dresses,20.82,9.05,834,Netherlands,26-35
2250,2394,172,246,5,28.98,20.29,8.69,30.00%,1,101.43,2025-05-13,App Mobile,App Mobile,Relaxed Satin Dress,Dresses,28.98,19.66,398,Spain,16-25
2251,1631,986,195,5,16.69,16.69,0.00,0.00%,0,83.45,2025-05-23,App Mobile,App Mobile,Vintage Crew Tee,T-Shirts,16.69,8.37,735,France,26-35


In [28]:
master['profit'] = (master['unit_price'] - master['cost_price']) * master['quantity']
master

,item_id,sale_id,product_id,quantity,original_price,unit_price,discount_applied,discount_percent,discounted,revenue,...,channel,channel_campaigns,product_name,category,catalog_price,cost_price,customer_id,country,age_range,profit
0,2270,658,403,1,81.80,81.80,0.00,0.00%,0,81.80,...,App Mobile,App Mobile,Elegant Satin Dress,Dresses,81.80,45.12,835,Portugal,46-55,36.68
1,1170,336,284,1,81.79,81.79,0.00,0.00%,0,81.79,...,E-commerce,Website Banner,Essential Cotton Shoes,Shoes,81.79,35.02,790,France,16-25,46.77
2,2496,1255,71,1,80.76,80.76,0.00,0.00%,0,80.76,...,App Mobile,App Mobile,Modern Ribbed Trousers,Pants,80.76,51.01,464,Germany,36-45,29.75
3,1273,331,98,1,78.52,78.52,0.00,0.00%,0,78.52,...,App Mobile,App Mobile,Modern Boxy Shoes,Shoes,78.52,41.48,100,Italy,26-35,37.04
4,1829,1079,98,1,78.52,78.52,0.00,0.00%,0,78.52,...,App Mobile,App Mobile,Modern Boxy Shoes,Shoes,78.52,41.48,837,Germany,46-55,37.04
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2248,516,1300,334,5,20.82,20.82,0.00,0.00%,0,104.10,...,E-commerce,Website Banner,Tailored Ribbed Dress,Dresses,20.82,9.05,147,Germany,36-45,58.85
2249,1240,727,334,5,20.82,20.82,0.00,0.00%,0,104.10,...,E-commerce,Website Banner,Tailored Ribbed Dress,Dresses,20.82,9.05,834,Netherlands,26-35,58.85
2250,2394,172,246,5,28.98,20.29,8.69,30.00%,1,101.43,...,App Mobile,App Mobile,Relaxed Satin Dress,Dresses,28.98,19.66,398,Spain,16-25,3.15
2251,1631,986,195,5,16.69,16.69,0.00,0.00%,0,83.45,...,App Mobile,App Mobile,Vintage Crew Tee,T-Shirts,16.69,8.37,735,France,26-35,41.60


In [29]:
master['profit_margin_pct'] = (master['profit'] / master['revenue']) * 100
master

,item_id,sale_id,product_id,quantity,original_price,unit_price,discount_applied,discount_percent,discounted,revenue,...,channel_campaigns,product_name,category,catalog_price,cost_price,customer_id,country,age_range,profit,profit_margin_pct
0,2270,658,403,1,81.80,81.80,0.00,0.00%,0,81.80,...,App Mobile,Elegant Satin Dress,Dresses,81.80,45.12,835,Portugal,46-55,36.68,44.841076
1,1170,336,284,1,81.79,81.79,0.00,0.00%,0,81.79,...,Website Banner,Essential Cotton Shoes,Shoes,81.79,35.02,790,France,16-25,46.77,57.183030
2,2496,1255,71,1,80.76,80.76,0.00,0.00%,0,80.76,...,App Mobile,Modern Ribbed Trousers,Pants,80.76,51.01,464,Germany,36-45,29.75,36.837543
3,1273,331,98,1,78.52,78.52,0.00,0.00%,0,78.52,...,App Mobile,Modern Boxy Shoes,Shoes,78.52,41.48,100,Italy,26-35,37.04,47.172695
4,1829,1079,98,1,78.52,78.52,0.00,0.00%,0,78.52,...,App Mobile,Modern Boxy Shoes,Shoes,78.52,41.48,837,Germany,46-55,37.04,47.172695
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2248,516,1300,334,5,20.82,20.82,0.00,0.00%,0,104.10,...,Website Banner,Tailored Ribbed Dress,Dresses,20.82,9.05,147,Germany,36-45,58.85,56.532181
2249,1240,727,334,5,20.82,20.82,0.00,0.00%,0,104.10,...,Website Banner,Tailored Ribbed Dress,Dresses,20.82,9.05,834,Netherlands,26-35,58.85,56.532181
2250,2394,172,246,5,28.98,20.29,8.69,30.00%,1,101.43,...,App Mobile,Relaxed Satin Dress,Dresses,28.98,19.66,398,Spain,16-25,3.15,3.105590
2251,1631,986,195,5,16.69,16.69,0.00,0.00%,0,83.45,...,App Mobile,Vintage Crew Tee,T-Shirts,16.69,8.37,735,France,26-35,41.60,49.850210


In [30]:
master[['revenue','profit','profit_margin_pct']].describe()

,revenue,profit,profit_margin_pct
count,2253.000000,2253.000000,2253.000000
mean,143.913298,62.651354,43.198141
std,82.153406,39.885720,11.088816
min,13.510000,0.160000,0.431732
25%,74.700000,29.980000,35.890118
50%,130.500000,56.550000,43.241491
75%,198.750000,87.900000,52.039236
max,403.800000,219.150000,59.858068


In [31]:
master2 = master[['discounted','discount_percent','quantity','revenue','profit','profit_margin_pct']].copy()

In [32]:
master2

,discounted,discount_percent,quantity,revenue,profit,profit_margin_pct
0,0,0.00%,1,81.80,36.68,44.841076
1,0,0.00%,1,81.79,46.77,57.183030
2,0,0.00%,1,80.76,29.75,36.837543
3,0,0.00%,1,78.52,37.04,47.172695
4,0,0.00%,1,78.52,37.04,47.172695
...,...,...,...,...,...,...
2248,0,0.00%,5,104.10,58.85,56.532181
2249,0,0.00%,5,104.10,58.85,56.532181
2250,1,30.00%,5,101.43,3.15,3.105590
2251,0,0.00%,5,83.45,41.60,49.850210


In [33]:
master2.groupby('discounted')[['quantity', 'revenue', 'profit', 'profit_margin_pct']].agg(['sum', 'mean'])

quantity              revenue                 profit             \
                sum      mean        sum        mean        sum       mean   
discounted                                                                   
0              6082  2.994584  300663.83  148.037336  134898.76  66.419872   
1               633  2.851351   23572.83  106.183919    6254.74  28.174505   

           profit_margin_pct             
                         sum       mean  
discounted                               
0               91446.183094  45.025201  
1                5879.228460  26.483011

In [34]:
from scipy.stats import ttest_ind

In [35]:
discounted_profit = master2.loc[master2['discounted'] == 1, 'profit']

non_discounted_profit = master2.loc[master2['discounted'] == 0, 'profit']

t_stat, p_value = ttest_ind(discounted_profit,non_discounted_profit, equal_var=False)

print('t-statistics:', t_stat)
print('p-value:', p_value) 

t-statistics: -20.141751870614026
p-value: 1.2873599040272782e-60


In [36]:
master2.groupby('discount_percent')[['quantity', 'revenue', 'profit', 'profit_margin_pct']].agg(['sum', 'mean'])

quantity              revenue                 profit  \
                      sum      mean        sum        mean        sum   
discount_percent                                                        
0.00%                6082  2.994584  300663.83  148.037336  134898.76   
10.00%                181  2.873016    7747.96  122.983492    2908.28   
30.00%                452  2.842767   15824.87   99.527484    3346.46   

                            profit_margin_pct             
                       mean               sum       mean  
discount_percent                                          
0.00%             66.419872      91446.183094  45.025201  
10.00%            46.163175       2401.463980  38.118476  
30.00%            21.046918       3477.764481  21.872733

Higher discounts significantly reduce profit and profit margins, while showing no evidence of increasing average purchase quantity.

In [37]:
from scipy.stats import f_oneway

In [38]:
q0 = master2.loc[master2['discount_percent'] == '0.00%', 'quantity']
q10 = master2.loc[master2['discount_percent'] == '10.00%', 'quantity']
q30 = master2.loc[master2['discount_percent'] == '30.00%', 'quantity']

f_stat, p_value = f_oneway(q0, q10, q30)

print("F-statistic:", f_stat)
print("p-value:", p_value)

F-statistic: 1.0273829497309985
p-value: 0.35811034384431817


# finding

- higher discount significantly reduce profitablility without increasing purchase quantity volume.

- the analysis showed that increasing discount rate from 0% to 30% led to a decline in profit margin. average profit margin
dropped from 45% at 0% discount and 38.1% at 10% discount reaching 21.9% at 30% discount. however the average quantity purchased didn't change across the discount levels with ANOVA test showing p=0.358 finding no statistically significant difference in purchase quantity between discount groups. 

- some heavily discounted products were sold at extrem low profit margin closer to cost, however the dataset doesn't include invetory age, and storage cost info which does not allow us to conclude if these discounts were stratigically justified.

# H3

is nventory allocation aligned with demand?

In [39]:
stock

,country,product_id,stock_quantity
0,France,1,61
1,France,2,24
2,France,3,81
3,France,4,70
4,France,5,30
...,...,...,...
995,Germany,496,1
996,Germany,497,1
997,Germany,498,1
998,Germany,499,1


In [40]:
master

,item_id,sale_id,product_id,quantity,original_price,unit_price,discount_applied,discount_percent,discounted,revenue,...,channel_campaigns,product_name,category,catalog_price,cost_price,customer_id,country,age_range,profit,profit_margin_pct
0,2270,658,403,1,81.80,81.80,0.00,0.00%,0,81.80,...,App Mobile,Elegant Satin Dress,Dresses,81.80,45.12,835,Portugal,46-55,36.68,44.841076
1,1170,336,284,1,81.79,81.79,0.00,0.00%,0,81.79,...,Website Banner,Essential Cotton Shoes,Shoes,81.79,35.02,790,France,16-25,46.77,57.183030
2,2496,1255,71,1,80.76,80.76,0.00,0.00%,0,80.76,...,App Mobile,Modern Ribbed Trousers,Pants,80.76,51.01,464,Germany,36-45,29.75,36.837543
3,1273,331,98,1,78.52,78.52,0.00,0.00%,0,78.52,...,App Mobile,Modern Boxy Shoes,Shoes,78.52,41.48,100,Italy,26-35,37.04,47.172695
4,1829,1079,98,1,78.52,78.52,0.00,0.00%,0,78.52,...,App Mobile,Modern Boxy Shoes,Shoes,78.52,41.48,837,Germany,46-55,37.04,47.172695
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2248,516,1300,334,5,20.82,20.82,0.00,0.00%,0,104.10,...,Website Banner,Tailored Ribbed Dress,Dresses,20.82,9.05,147,Germany,36-45,58.85,56.532181
2249,1240,727,334,5,20.82,20.82,0.00,0.00%,0,104.10,...,Website Banner,Tailored Ribbed Dress,Dresses,20.82,9.05,834,Netherlands,26-35,58.85,56.532181
2250,2394,172,246,5,28.98,20.29,8.69,30.00%,1,101.43,...,App Mobile,Relaxed Satin Dress,Dresses,28.98,19.66,398,Spain,16-25,3.15,3.105590
2251,1631,986,195,5,16.69,16.69,0.00,0.00%,0,83.45,...,App Mobile,Vintage Crew Tee,T-Shirts,16.69,8.37,735,France,26-35,41.60,49.850210


In [41]:
country_demand = master.groupby('country')['quantity'].sum().reset_index(name='unit_sold')
country_demand

,country,unit_sold
0,France,1469
1,Germany,1579
2,Italy,1230
3,Netherlands,989
4,Portugal,605
5,Spain,843


In [42]:
country_stock = stock.groupby('country')['stock_quantity'].sum().reset_index(name='stock_available')
country_stock

,country,stock_available
0,France,24136
1,Germany,500


In [43]:
h3 = country_demand.merge(country_stock, on='country', how='left')
h3

,country,unit_sold,stock_available
0,France,1469,24136.0
1,Germany,1579,500.0
2,Italy,1230,NaN
3,Netherlands,989,NaN
4,Portugal,605,NaN
5,Spain,843,NaN


In [44]:
h3['sell_through_ratio'] = (h3['unit_sold'] / h3['stock_available'])
h3.sort_values('sell_through_ratio', ascending=False)

,country,unit_sold,stock_available,sell_through_ratio
1,Germany,1579,500.0,3.158000
0,France,1469,24136.0,0.060863
2,Italy,1230,NaN,NaN
3,Netherlands,989,NaN,NaN
4,Portugal,605,NaN,NaN
5,Spain,843,NaN,NaN


# findings

- sales analysis showed that germany generated the highest demand among all countries with 1579 pieces sold over the period of
  months and a half, while the availble inventory is 500 pieces, on the other hand france has a demand of 1469 while having 24136
  pieces in invetory which suggest that france may be acting as a central warehouse between the supplied countries to reduce the
  costs but the dataset doesn't include enough data to prove this assumption.

- the finding also suggest that inventory allocation may not be fully reflecting customer demand pattern. since germany generating the highest sales volume despite the low inventory.


# limitations

the finding should be considered with other factors related to the operational process, there is no data that supports the claim that france is the central hub, nor any data for delivery costs and warehouse costs. 

# Exploratory Findings

# H4

Some product categories generate disproportionately higher profit than revenue.

In [45]:
master

,item_id,sale_id,product_id,quantity,original_price,unit_price,discount_applied,discount_percent,discounted,revenue,...,channel_campaigns,product_name,category,catalog_price,cost_price,customer_id,country,age_range,profit,profit_margin_pct
0,2270,658,403,1,81.80,81.80,0.00,0.00%,0,81.80,...,App Mobile,Elegant Satin Dress,Dresses,81.80,45.12,835,Portugal,46-55,36.68,44.841076
1,1170,336,284,1,81.79,81.79,0.00,0.00%,0,81.79,...,Website Banner,Essential Cotton Shoes,Shoes,81.79,35.02,790,France,16-25,46.77,57.183030
2,2496,1255,71,1,80.76,80.76,0.00,0.00%,0,80.76,...,App Mobile,Modern Ribbed Trousers,Pants,80.76,51.01,464,Germany,36-45,29.75,36.837543
3,1273,331,98,1,78.52,78.52,0.00,0.00%,0,78.52,...,App Mobile,Modern Boxy Shoes,Shoes,78.52,41.48,100,Italy,26-35,37.04,47.172695
4,1829,1079,98,1,78.52,78.52,0.00,0.00%,0,78.52,...,App Mobile,Modern Boxy Shoes,Shoes,78.52,41.48,837,Germany,46-55,37.04,47.172695
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2248,516,1300,334,5,20.82,20.82,0.00,0.00%,0,104.10,...,Website Banner,Tailored Ribbed Dress,Dresses,20.82,9.05,147,Germany,36-45,58.85,56.532181
2249,1240,727,334,5,20.82,20.82,0.00,0.00%,0,104.10,...,Website Banner,Tailored Ribbed Dress,Dresses,20.82,9.05,834,Netherlands,26-35,58.85,56.532181
2250,2394,172,246,5,28.98,20.29,8.69,30.00%,1,101.43,...,App Mobile,Relaxed Satin Dress,Dresses,28.98,19.66,398,Spain,16-25,3.15,3.105590
2251,1631,986,195,5,16.69,16.69,0.00,0.00%,0,83.45,...,App Mobile,Vintage Crew Tee,T-Shirts,16.69,8.37,735,France,26-35,41.60,49.850210


In [46]:
master['category'].nunique()

5

In [47]:
category_profit = (master.groupby('category').agg(revenue=('revenue','sum'),profit=('profit','sum')).reset_index())

category_profit

,category,revenue,profit
0,Dresses,68390.64,29846.84
1,Pants,53802.66,24049.94
2,Shoes,70074.00,30473.47
3,Sleepwear,62276.58,26000.58
4,T-Shirts,69692.78,30782.67


In [48]:
category_profit['profit_margin_pct'] = (category_profit['profit'] / category_profit['revenue']) * 100

category_profit.sort_values('profit_margin_pct',ascending=False)

,category,revenue,profit,profit_margin_pct
1,Pants,53802.66,24049.94,44.700281
4,T-Shirts,69692.78,30782.67,44.169095
0,Dresses,68390.64,29846.84,43.641703
2,Shoes,70074.00,30473.47,43.487556
3,Sleepwear,62276.58,26000.58,41.750173


In [49]:
pants = master[master['category'] == 'Pants']['profit_margin_pct']
tshirt = master[master['category'] == 'T-Shirts']['profit_margin_pct']
dresses = master[master['category'] == 'Dresses']['profit_margin_pct']
shoes = master[master['category'] == 'Shoes']['profit_margin_pct']
sleepwear = master[master['category'] == 'Sleepwear']['profit_margin_pct']

f_stat, p_value = f_oneway(pants, tshirt, dresses, shoes, sleepwear)

print("F-statistics:", f_stat)
print("p-value:", p_value)

F-statistics: 6.202968971063424
p-value: 5.815320227810817e-05


# findings

- although ANOVA found a statistically significant difference in proft margin across the categories with F = 6.20, and p < 0.05 yet the practical difference between categories is relativley small at approx 3 percent. therefore category not a major driver for profitablility. 

- No meaningful diffrence found across product categories 

# H5

Customer age segments differ significantly in profitability.

In [50]:
master

,item_id,sale_id,product_id,quantity,original_price,unit_price,discount_applied,discount_percent,discounted,revenue,...,channel_campaigns,product_name,category,catalog_price,cost_price,customer_id,country,age_range,profit,profit_margin_pct
0,2270,658,403,1,81.80,81.80,0.00,0.00%,0,81.80,...,App Mobile,Elegant Satin Dress,Dresses,81.80,45.12,835,Portugal,46-55,36.68,44.841076
1,1170,336,284,1,81.79,81.79,0.00,0.00%,0,81.79,...,Website Banner,Essential Cotton Shoes,Shoes,81.79,35.02,790,France,16-25,46.77,57.183030
2,2496,1255,71,1,80.76,80.76,0.00,0.00%,0,80.76,...,App Mobile,Modern Ribbed Trousers,Pants,80.76,51.01,464,Germany,36-45,29.75,36.837543
3,1273,331,98,1,78.52,78.52,0.00,0.00%,0,78.52,...,App Mobile,Modern Boxy Shoes,Shoes,78.52,41.48,100,Italy,26-35,37.04,47.172695
4,1829,1079,98,1,78.52,78.52,0.00,0.00%,0,78.52,...,App Mobile,Modern Boxy Shoes,Shoes,78.52,41.48,837,Germany,46-55,37.04,47.172695
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2248,516,1300,334,5,20.82,20.82,0.00,0.00%,0,104.10,...,Website Banner,Tailored Ribbed Dress,Dresses,20.82,9.05,147,Germany,36-45,58.85,56.532181
2249,1240,727,334,5,20.82,20.82,0.00,0.00%,0,104.10,...,Website Banner,Tailored Ribbed Dress,Dresses,20.82,9.05,834,Netherlands,26-35,58.85,56.532181
2250,2394,172,246,5,28.98,20.29,8.69,30.00%,1,101.43,...,App Mobile,Relaxed Satin Dress,Dresses,28.98,19.66,398,Spain,16-25,3.15,3.105590
2251,1631,986,195,5,16.69,16.69,0.00,0.00%,0,83.45,...,App Mobile,Vintage Crew Tee,T-Shirts,16.69,8.37,735,France,26-35,41.60,49.850210


In [51]:
age_profit = master.groupby('age_range').agg(revenue=('revenue', 'sum'),
                                           profit=('profit', 'sum'),
                                           quantity=('quantity', 'sum'),
                                            margin=('profit_margin_pct', 'mean')).reset_index()
age_profit

,age_range,revenue,profit,quantity,margin
0,16-25,65918.37,28499.98,1360,42.982581
1,26-35,69466.31,30319.57,1409,43.237808
2,36-45,68371.81,29736.97,1422,43.124646
3,46-55,62635.41,26850.12,1309,42.675600
4,56-65,57844.76,25746.86,1215,44.000567


In [52]:
age_profit['profit_margin_pct'] = (age_profit['profit'] / age_profit['revenue']) * 100
age_profit.sort_values('profit_margin_pct', ascending=False)

,age_range,revenue,profit,quantity,margin,profit_margin_pct
4,56-65,57844.76,25746.86,1215,44.000567,44.510272
1,26-35,69466.31,30319.57,1409,43.237808,43.646438
2,36-45,68371.81,29736.97,1422,43.124646,43.493027
0,16-25,65918.37,28499.98,1360,42.982581,43.235262
3,46-55,62635.41,26850.12,1309,42.675600,42.867317


# Finding 

- profitability is relatively consistent acreoss age groups suggesting that age group is not a major driver of profit margin. 

- no meaningful difference found across age groups

In [59]:
master.to_csv('master.csv', index=False)

In [55]:
master.columns

Index(['item_id', 'sale_id', 'product_id', 'quantity', 'original_price',
       'unit_price', 'discount_applied', 'discount_percent', 'discounted',
       'revenue', 'sale_date', 'channel', 'channel_campaigns', 'product_name',
       'category', 'catalog_price', 'cost_price', 'customer_id', 'country',
       'age_range', 'profit', 'profit_margin_pct'],
      dtype='object')